<a href="https://colab.research.google.com/github/Phani-ISB/Reconciliation_Modules/blob/main/1_Inter_Company_Reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing all the required libraries

In [198]:
pip install -q rapidfuzz

In [199]:
import numpy as np
import pandas as pd
from datetime import timedelta
from rapidfuzz import fuzz

# Load the Data

In [200]:
df = pd.read_excel("/content/IC_InputData (5) - Copy.xlsx")

In [201]:
df.head()

,Entity,PartnerEntity,AccountingStandard,Year,Period,TransactionType,TransactionDate,FlexDate1,FlexDate2,KeyIdentifier1,...,DeferredTax,DepreciationForTheYear,OpeningAccumulatedDepreciation,WhetherInClosingStock,PercentOrAmountInClosingStock,Custom1,Custom2,Custom3,GL_Code,GL_Description
0,I053,I052,Algeria GAAP,2023,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable
1,I053,I052,Algeria GAAP,2023,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,0,0,0,False,0,NaN,NaN,NaN,1500001,Intercompany Loans Payable
2,I053,I052,Algeria GAAP,2023,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable
3,I053,I052,Algeria GAAP,2023,Year,IN,2023-06-21,2023-06-21,2023-11-26,910000007,...,0,0,0,False,0,NaN,NaN,NaN,1500001,Intercompany Loans Payable
4,I053,I052,Algeria GAAP,2023,Year,IN,2023-04-13,2023-04-13,2024-01-09,910000042,...,0,0,0,False,0,NaN,NaN,NaN,1300151,IC Loan Payable


In [202]:
# Convert Date and Numeric Columns
df['TransactionDate'] =pd.to_datetime(df['TransactionDate'])
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

In [203]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 328 entries, 0 to 327
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Entity                          328 non-null    object        
 1   PartnerEntity                   328 non-null    object        
 2   AccountingStandard              328 non-null    object        
 3   Year                            328 non-null    int64         
 4   Period                          328 non-null    object        
 5   TransactionType                 328 non-null    object        
 6   TransactionDate                 328 non-null    datetime64[ns]
 7   FlexDate1                       328 non-null    datetime64[ns]
 8   FlexDate2                       328 non-null    datetime64[ns]
 9   KeyIdentifier1                  328 non-null    int64         
 10  KeyIdentifier2                  326 non-null    object        
 11  KeyIde

# EDA

In [204]:
# Entity Summaries
entity_summary = df.groupby('Entity')['Amount'].sum().reset_index()

In [205]:
entity_summary

,Entity,Amount
0,I051,972498.82
1,I052,9209515.41
2,I053,-12872.41
3,I054,42787.00
4,I055,-184879.00
5,I056,-10027049.99
6,I057,-0.05
7,I058,23458999.14
8,I059,-10720459.59
9,I060,-12738538.88


In [206]:
# Total Amounts Tally
entity_summary['Amount'].sum()

np.float64(0.4499999936670065)

In [207]:
#Currency Mismatches

currency_check = df.groupby(['Entity','PartnerEntity','TransactionCurrency']).size().unstack(fill_value=0)
currency_check

TransactionCurrency   INR
Entity PartnerEntity     
I051   I052             1
I052   I051             1
       I053            70
       I054             9
       I055            10
       I056             8
       I058             2
I053   I052            63
       I055            13
I054   I052             3
I055   I052            10
       I053            13
I056   I052            12
I057   I058            15
I058   I052             6
       I057            18
       I059             9
       I060             8
I059   I058            15
       I060            15
I060   I054             2
       I058             8
       I059            17

In [208]:
# Duplicates Check
print("Transaction Duplicates:", df.duplicated().sum())

Transaction Duplicates: 0


In [209]:
# Normalize text
df['Narration'] = (
    df['Narration1'].fillna('') + ' ' + df['Narration3'].fillna('')
).str.lower().str.strip()

# Ensure datetime
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

# Standard Reconciliation

In [210]:
# Initial columns
df['Recon_Status'] = 'Unmatched'
df['Rule_Applied'] = None
df['Recon_ID'] = None

In [211]:
# Matching Functions

AMT_TOLERANCE = 1.0
FUZZY_THRESHOLD = 70

def amount_match(a, b, tol):
    return abs(a + b) <= tol

def date_match(d1, d2, days):
    if days is None:
        return True
    return abs((d1 - d2).days) <= days

def narration_match(n1, n2, fuzzy):
    if fuzzy is None:
        return True
    if not fuzzy:
        return n1 == n2
    return fuzz.partial_ratio(n1, n2) >= FUZZY_THRESHOLD

In [212]:
# Sequential Rules
rules = [
    {"name": "1. Narration Exact + Date Exact", "dt": 0, "fuzzy": False},
    {"name": "2. Narration Fuzzy + Date Exact", "dt": 0, "fuzzy": True},
    {"name": "3. Narration Exact + Date Range", "dt": 5, "fuzzy": False},
    {"name": "4. Narration Fuzzy + Date Range", "dt": 5, "fuzzy": True},
    {"name": "5. Only Date Exact", "dt": 0, "fuzzy": None},
    {"name": "6. Amount Match Only", "dt": None, "fuzzy": None}
]

In [213]:
recon_group_id = 1

for rule in rules:

    unmatched = df[df['Recon_Status'] == 'Unmatched']

    for i, row in unmatched.iterrows():

        if df.loc[i, 'Recon_Status'] != 'Unmatched':
            continue

        # Cache values (faster)
        entity = row['Entity']
        partner = row['PartnerEntity']
        amt = row['Amount']
        date = row['TransactionDate']
        narr = row['Narration']

        # Candidate filtering
        candidates = unmatched[
            (unmatched['Entity'] == partner) &
            (unmatched['PartnerEntity'] == entity) &
            (unmatched.index != i)
        ]

        for j, crow in candidates.iterrows():

            if df.loc[j, 'Recon_Status'] != 'Unmatched':
                continue

            # Cache candidate values
            c_amt = crow['Amount']
            c_date = crow['TransactionDate']
            c_narr = crow['Narration']

            # Amount condition
            if not amount_match(amt, c_amt, AMT_TOLERANCE):
                continue

            # Date condition
            if not date_match(date, c_date, rule['dt']):
                continue

            # Narration condition
            if not narration_match(narr, c_narr, rule['fuzzy']):
                continue

            # MATCH FOUND
            recon_id = f"REC_{recon_group_id}"

            df.loc[[i, j], 'Recon_Status'] = 'Matched'
            df.loc[[i, j], 'Rule_Applied'] = rule['name']
            df.loc[[i, j], 'Recon_ID'] = recon_id

            recon_group_id += 1
            break

In [214]:
# Check for amount differences in the reconciled group
df['Amt_Diff'] = df.groupby('Recon_ID')['Amount'].transform('sum')

In [215]:
# MAtched and Unmatched results

matched_df = df[df['Recon_Status'] == 'Matched']
unmatched_df = df[df['Recon_Status'] == 'Unmatched']

In [216]:
matched_df

,Entity,PartnerEntity,AccountingStandard,Year,Period,TransactionType,TransactionDate,FlexDate1,FlexDate2,KeyIdentifier1,...,Custom1,Custom2,Custom3,GL_Code,GL_Description,Narration,Recon_Status,Rule_Applied,Recon_ID,Amt_Diff
0,I053,I052,Algeria GAAP,2023,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 ici052,Matched,6. Amount Match Only,REC_122,0.0
1,I053,I052,Algeria GAAP,2023,Year,RR,2023-06-21,2023-06-21,2024-01-09,900000012,...,NaN,NaN,NaN,1500001,Intercompany Loans Payable,ic loan 1300151,Matched,6. Amount Match Only,REC_123,0.0
4,I053,I052,Algeria GAAP,2023,Year,IN,2023-04-13,2023-04-13,2024-01-09,910000042,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 ici052,Matched,2. Narration Fuzzy + Date Exact,REC_22,0.0
5,I053,I052,Algeria GAAP,2023,Year,IN,2023-04-13,2023-04-13,2024-01-09,910000042,...,NaN,NaN,NaN,1500001,Intercompany Loans Payable,i052 1300151,Matched,2. Narration Fuzzy + Date Exact,REC_23,0.0
6,I053,I052,Algeria GAAP,2023,Year,IN,2023-04-28,2023-04-28,2023-11-26,910000008,...,NaN,NaN,NaN,1300151,IC Loan Payable,i052 ici052,Matched,2. Narration Fuzzy + Date Exact,REC_24,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,I059,I060,Algeria GAAP,2023,Year,RR,2023-11-07,2023-11-07,2023-11-13,900000001,...,NaN,NaN,NaN,2100026,IntercompanyLoansR,ic loan repaymnt 2020540,Matched,2. Narration Fuzzy + Date Exact,REC_108,0.0
322,I059,I060,Algeria GAAP,2023,Year,IN,2023-03-31,2023-03-31,2024-01-22,910000042,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,i060 2100151,Matched,2. Narration Fuzzy + Date Exact,REC_101,0.0
323,I058,I060,Algeria GAAP,2023,Year,IN,2023-03-31,2023-03-31,2024-01-21,910000008,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,i060 2100151,Matched,2. Narration Fuzzy + Date Exact,REC_99,0.0
324,I058,I060,Algeria GAAP,2023,Year,IN,2023-03-31,2023-03-31,2024-01-21,910000008,...,NaN,NaN,NaN,2100151,IC Loan Receivable,i060 ici060,Matched,2. Narration Fuzzy + Date Exact,REC_100,0.0


In [217]:
# Reconciliation matrix
# Aggregate by Entity → PartnerEntity
matrix = matched_df.groupby(['Entity', 'PartnerEntity'])['Amount'].sum().reset_index()

# Pivot
recon_matrix = matrix.pivot(
    index='Entity',
    columns='PartnerEntity',
    values='Amount'
).fillna(0)

recon_matrix

PartnerEntity,I051,I052,I053,I054,I055,I056,I057,I058,I059,I060
Entity,,,,,,,,,,
I051,0.00,972498.82,0.0,0.0,0.0,0.0,0.000000e+00,0.00,0.00,0.00
I052,-972498.81,0.00,79345.0,-42787.0,102515.0,10027050.0,0.000000e+00,0.12,0.00,0.00
I053,0.00,-79346.00,0.0,0.0,82364.0,0.0,0.000000e+00,0.00,0.00,0.00
I054,0.00,42787.00,0.0,0.0,0.0,0.0,0.000000e+00,0.00,0.00,0.00
I055,0.00,-102515.00,-82364.0,0.0,0.0,0.0,0.000000e+00,0.00,0.00,0.00
I056,0.00,-10027049.99,0.0,0.0,0.0,0.0,0.000000e+00,0.00,0.00,0.00
I057,0.00,0.00,0.0,0.0,0.0,0.0,0.000000e+00,-0.05,0.00,0.00
I058,0.00,-0.12,0.0,0.0,0.0,0.0,-1.164153e-10,0.00,19804343.00,3654656.07
I059,0.00,0.00,0.0,0.0,0.0,0.0,0.000000e+00,-19804343.00,0.00,9083883.41


In [218]:
# Summary of Unmatched transactions
matrix = unmatched_df.groupby(['Entity', 'PartnerEntity'])['Amount'].sum().reset_index()

# Pivot
unrecon_matrix = matrix.pivot(
    index='Entity',
    columns='PartnerEntity',
    values='Amount'
).fillna(0)

unrecon_matrix

PartnerEntity,I052,I053,I054,I055,I056,I057,I058,I059,I060
Entity,,,,,,,,,
I052,0.00,15891.1,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I053,-15890.41,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I055,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I056,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I057,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I058,0.00,0.0,0.0,0.0,0.0,0.19,0.0,0.0,0.0
I059,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
I060,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0


In [219]:
unmatched_df[unmatched_df['Entity'] == 'I052']

,Entity,PartnerEntity,AccountingStandard,Year,Period,TransactionType,TransactionDate,FlexDate1,FlexDate2,KeyIdentifier1,...,Custom1,Custom2,Custom3,GL_Code,GL_Description,Narration,Recon_Status,Rule_Applied,Recon_ID,Amt_Diff
78,I052,I053,Algeria GAAP,2023,Year,IC,2023-02-02,2023-02-02,2023-05-27,850000028,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3419 2000031,Unmatched,None,None,NaN
79,I052,I053,Algeria GAAP,2023,Year,IC,2023-02-02,2023-02-02,2023-05-27,850000034,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3419 2000031,Unmatched,None,None,NaN
80,I052,I053,Algeria GAAP,2023,Year,IC,2023-02-02,2023-02-02,2023-05-27,850000037,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3419 2000031,Unmatched,None,None,NaN
81,I052,I053,Algeria GAAP,2023,Year,IC,2023-02-02,2023-02-02,2023-05-29,850000056,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3419 2000031,Unmatched,None,None,NaN
82,I052,I053,Algeria GAAP,2023,Year,IC,2023-02-02,2023-03-31,2023-08-23,850000060,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3419 2000031,Unmatched,None,None,NaN
84,I052,I053,Algeria GAAP,2023,Year,IC,2023-03-31,2023-03-31,2023-05-27,850000026,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3442 2000031,Unmatched,None,None,NaN
85,I052,I053,Algeria GAAP,2023,Year,IC,2023-03-31,2023-03-31,2023-05-29,850000054,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3442 2000031,Unmatched,None,None,NaN
86,I052,I053,Algeria GAAP,2023,Year,IC,2023-03-31,2023-03-31,2023-08-23,850000061,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,je-rrpl-3442 2000031,Unmatched,None,None,NaN
93,I052,I053,Algeria GAAP,2023,Year,IC,2023-03-31,2023-03-31,2024-01-11,850000078,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,aje 3100005,Unmatched,None,None,NaN
96,I052,I053,Algeria GAAP,2023,Year,IN,2023-03-31,2023-03-31,2024-02-27,910000044,...,NaN,NaN,NaN,1200007,Accounts Payable - Intercompany,i053 2100152,Unmatched,None,None,NaN
